# HDB Resale Flat Prices — ETL (Jan 2012 to Dec 2017)

The pipeline runs in **three stages**, one per layer of the medallion architecture. Each stage reads only the
files written by the previous stage, so any stage can be re-run on its own (re-run its "Run the stage" cell).

| Stage | What it does | Reads | Writes (output group) |
|---|---|---|---|
| **1. `src_to_raw`** | Find the datasets on data.gov.sg and download them as-is | data.gov.sg API | `data/raw/` (**Raw**) |
| **2. `raw_to_silver`** | Combine, profile, validate, clean, remaining lease, price anomalies | `data/raw/` | `data/silver/cleaned.csv` (**Cleaned**), `data/silver/quarantined.csv` (**Quarantined**) |
| **3. `silver_to_gold`** | Resale identifier, then hash it | `data/silver/cleaned.csv` | `data/gold/transformed.csv` (**Transformed**), `data/gold/hashed.csv` (**Hashed**) |

Every step is logged with its stage name, to the cell output and to `logs/pipeline.log`, so a failure shows
immediately which stage it happened in. Settings are in `config.json`.

**How to run:** Kernel → Restart & Run All. Running all cells runs the three stages in order.

## 0. Settings and logging
All settings come from `config.json`, so nothing is hard-coded in the functions below. Logging is set up once:
each stage has its own logger, so every log line starts with the stage name.

In [1]:
import hashlib
import hmac
import json
import logging
import os
import secrets
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

with open("config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)

# Source: data.gov.sg collection and API endpoints
COLLECTION_ID = CONFIG["source"]["collection_id"]
COLLECTION_URL = CONFIG["source"]["collection_metadata_url"]
DATASET_URL = CONFIG["source"]["dataset_metadata_url"]
DOWNLOAD_URL = CONFIG["source"]["download_api_url"]

# Scope and reference set
START_MONTH = CONFIG["scope"]["start_month"]
END_MONTH = CONFIG["scope"]["end_month"]
REFERENCE_MONTH = CONFIG["scope"]["reference_month"]          # Jan 2012 is the authoritative set

# Validation rules
REFERENCE_COLUMNS = CONFIG["validation"]["reference_columns"]  # value must exist in Jan 2012
STOREY_PATTERN = CONFIG["validation"]["storey_pattern"]        # format taken from Jan 2012, e.g. "04 TO 06"
NUMERIC_COLUMNS = CONFIG["validation"]["numeric_columns"]      # must be positive numbers

# Remaining lease
LEASE_YEARS = CONFIG["lease"]["lease_years"]
LEASE_AS_OF = pd.Timestamp(CONFIG["lease"]["as_of_date"])

# Price anomalies
ANOMALY_GROUP = CONFIG["anomaly"]["group_columns"]
IQR_MULTIPLIER = CONFIG["anomaly"]["iqr_multiplier"]
MIN_GROUP_SIZE = CONFIG["anomaly"]["min_group_size"]

# Hashing: only the name of the environment variable is stored, never the key
HMAC_KEY_ENV = CONFIG["hashing"]["key_env_var"]

# Medallion layers and logs
RAW_DIR = Path(CONFIG["paths"]["raw"])        # bronze: source files as-is
SILVER_DIR = Path(CONFIG["paths"]["silver"])  # validated and cleaned (plus quarantined rows)
GOLD_DIR = Path(CONFIG["paths"]["gold"])      # business-ready: identifier added, identifier hashed
LOG_FILE = Path(CONFIG["paths"]["log_file"])
for folder in (RAW_DIR, SILVER_DIR, GOLD_DIR, LOG_FILE.parent):
    folder.mkdir(parents=True, exist_ok=True)

# Logging: every line shows the time, the stage and the level; written to the notebook and to the log file
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)-14s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_FILE, encoding="utf-8")],
    force=True,
)
log_src_to_raw = logging.getLogger("src_to_raw")
log_raw_to_silver = logging.getLogger("raw_to_silver")
log_silver_to_gold = logging.getLogger("silver_to_gold")


def run_stage(name: str, stage):
    """Run one pipeline stage: log its start, end and duration, or which stage failed and why."""
    log = logging.getLogger(name)
    log.info("Stage started")
    started = time.time()
    try:
        result = stage()
    except Exception:
        log.exception("Stage FAILED after %.1f s", time.time() - started)
        raise
    log.info("Stage finished in %.1f s", time.time() - started)
    return result


pd.set_option("display.max_columns", 30)

## 1. Stage `src_to_raw` — data.gov.sg → Raw
1. **Find the datasets.** Collection 189 on data.gov.sg is split into several files. Its metadata lists them and
   each file's metadata gives the months it covers. Every CSV file that overlaps the scope is selected, so no
   dataset IDs are hard-coded.
2. **Download each file** through the API (request the file, poll until it is ready, stream it to disk) and save it
   exactly as published. Files already in `data/raw/` are reused. The download goes to a temporary `.part` file that
   is renamed only when complete, so an interrupted download is never mistaken for a finished file.
3. **API failures:** rate limits (HTTP 429 or code 24), server errors (5xx) and network drops are retried with a
   longer wait each time; other errors such as 404 fail at once. These cases are tested in
   `tests/test_api_failures.py`.

In [2]:
def get_json(url: str, retries: int = 6) -> dict:
    """GET a data.gov.sg API URL.

    Temporary problems (rate limit, server error, network drop) are retried with a longer wait each time.
    Anything else, such as 404 Not Found, fails at once.
    """
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=60)
            problem = f"HTTP {response.status_code}"
        except (requests.ConnectionError, requests.Timeout) as error:
            response, problem = None, f"network error ({type(error).__name__})"
        if response is not None and response.status_code != 429 and response.status_code < 500:
            response.raise_for_status()                        # e.g. 404: a real error, do not retry
            body = response.json()
            if body.get("code") != 24:                         # code 24 = rate limited
                return body
            problem = "rate limited (code 24)"
        wait = 10 * attempt
        log_src_to_raw.warning("%s, retry %d of %d in %d s: %s", problem, attempt, retries, wait, url)
        time.sleep(wait)
    raise RuntimeError(f"data.gov.sg API still failing after {retries} attempts: {url}")


def find_datasets(collection_id: str, start: str, end: str) -> list[dict]:
    """CSV datasets in the collection whose coverage overlaps start..end, oldest first."""
    collection = get_json(COLLECTION_URL.format(collection_id=collection_id))["data"]["collectionMetadata"]
    datasets = []
    for dataset_id in collection["childDatasets"]:
        meta = get_json(DATASET_URL.format(dataset_id=dataset_id))["data"]
        first = meta["coverageStart"][:7]
        last = (meta.get("coverageEnd") or "9999-12")[:7]   # no end date = still being updated
        if meta.get("format") == "CSV" and first <= end and last >= start:
            datasets.append({"dataset_id": dataset_id, "name": meta["name"], "first_month": first, "last_month": last})
        else:
            log_src_to_raw.info("Skipped %s (%s to %s): outside %s..%s", dataset_id, first, last, start, end)
    log_src_to_raw.info("Collection %s: %d of %d datasets overlap %s..%s",
                        collection_id, len(datasets), len(collection["childDatasets"]), start, end)
    return sorted(datasets, key=lambda d: d["first_month"])


def download_dataset(dataset_id: str) -> Path:
    """Download one dataset as-is into RAW_DIR, unless it is already there."""
    path = RAW_DIR / f"{dataset_id}.csv"
    if path.exists():
        log_src_to_raw.info("%s already in raw, skipped download", path.name)
        return path
    base = DOWNLOAD_URL.format(dataset_id=dataset_id)
    get_json(f"{base}/initiate-download")
    for _ in range(24):
        url = (get_json(f"{base}/poll-download").get("data") or {}).get("url")
        if url:
            break
        time.sleep(5)
    else:
        raise RuntimeError(f"Download for {dataset_id} was not ready in time")
    # Write to a temporary .part file and rename it only when the download is complete,
    # so an interrupted download never leaves a half-written file that a later run would reuse.
    part = path.with_suffix(".part")
    started = time.time()
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(part, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    part.replace(path)
    log_src_to_raw.info("Downloaded %s (%.1f MB in %.1f s)", path.name, path.stat().st_size / 1e6,
                        time.time() - started)
    return path


def src_to_raw() -> pd.DataFrame:
    """Stage 1: find the in-scope datasets and make sure each one is in the raw layer."""
    datasets = find_datasets(COLLECTION_ID, START_MONTH, END_MONTH)
    for dataset in datasets:
        dataset["file"] = download_dataset(dataset["dataset_id"]).name
    return pd.DataFrame(datasets)

In [3]:
# Run the stage
raw_files = run_stage("src_to_raw", src_to_raw)
raw_files

2026-10-14 20:53:42 | src_to_raw     | INFO    | Stage started


2026-10-14 20:53:44 | src_to_raw     | INFO    | Skipped d_ebc5ab87086db484f88045b47411ebc5 (1990-01 to 1999-12): outside 2012-01..2017-12


2026-10-14 20:53:44 | src_to_raw     | INFO    | Collection 189: 4 of 5 datasets overlap 2012-01..2017-12


2026-10-14 20:53:44 | src_to_raw     | INFO    | d_43f493c6c50d54243cc1eab0df142d6a.csv already in raw, skipped download


2026-10-14 20:53:44 | src_to_raw     | INFO    | d_2d5ff9ea31397b66239f245f57751537.csv already in raw, skipped download


2026-10-14 20:53:44 | src_to_raw     | INFO    | d_ea9ed51da2787afaf8e51f827c304208.csv already in raw, skipped download


2026-10-14 20:53:44 | src_to_raw     | INFO    | d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv already in raw, skipped download


2026-10-14 20:53:44 | src_to_raw     | INFO    | Stage finished in 1.6 s


,dataset_id,name,first_month,last_month,file
0,d_43f493c6c50d54243cc1eab0df142d6a,"Resale Flat Prices (Based on Approval Date), 2...",2000-01,2012-02,d_43f493c6c50d54243cc1eab0df142d6a.csv
1,d_2d5ff9ea31397b66239f245f57751537,Resale Flat Prices (Based on Registration Date...,2012-03,2014-12,d_2d5ff9ea31397b66239f245f57751537.csv
2,d_ea9ed51da2787afaf8e51f827c304208,Resale Flat Prices (Based on Registration Date...,2015-01,2016-12,d_ea9ed51da2787afaf8e51f827c304208.csv
3,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,Resale flat prices based on registration date ...,2017-01,2026-10,d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv


## 2. Stage `raw_to_silver` — Raw → Cleaned + Quarantined
This stage reads the files in `data/raw/` and writes the silver layer. It is built from four groups of functions
(2.1 to 2.4), run together in 2.5; the results are shown in 2.6.

### 2.1 Load and combine
Every raw file is read **as text** (`dtype=str`, `keep_default_na=False`), so no value is changed or guessed on
load. The columns of the file containing Jan 2012 define the master schema; rows from 2012-01 to 2017-12 are taken
from every file, and each row keeps the name of the file it came from. The `remaining_lease` column that later files
add is not in the Jan 2012 schema, so it is left out and remaining lease is recalculated in 2.4.

In [4]:
def load_raw_files(raw_dir: Path) -> dict[str, pd.DataFrame]:
    """Read every raw CSV as text, keyed by file name."""
    frames = {}
    for path in sorted(raw_dir.glob("*.csv")):
        frames[path.name] = pd.read_csv(path, dtype=str, keep_default_na=False)
        log_raw_to_silver.info("Read %s: %d rows, %d columns", path.name, len(frames[path.name]),
                               frames[path.name].shape[1])
    if not frames:
        raise FileNotFoundError(f"No raw files in {raw_dir}: run stage src_to_raw first")
    return frames


def combine(frames: dict[str, pd.DataFrame], start: str, end: str, reference_month: str):
    """Stack the in-scope rows of every file using the reference file's columns."""
    reference_file = next(name for name, df in frames.items() if (df["month"] == reference_month).any())
    columns = list(frames[reference_file].columns)
    parts = [df.loc[df["month"].between(start, end), columns].assign(source_file=name)
             for name, df in frames.items()]
    master = pd.concat(parts, ignore_index=True)
    log_raw_to_silver.info("Master dataset: %d rows in %s..%s, schema from %s (%d columns)",
                           len(master), start, end, reference_file, len(columns))
    return master, columns

### 2.2 Profiling
Completeness, distinct values, numeric ranges, volumes, duplicates, and the values that do not appear in Jan 2012.

In [5]:
def column_profile(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: empty values, distinct values and the most common value."""
    return pd.DataFrame([{
        "column": c,
        "empty": int((df[c].str.strip() == "").sum()),
        "distinct": df[c].nunique(),
        "most_common": df[c].mode().iloc[0],
    } for c in df.columns])


def numeric_profile(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Summary statistics for columns that should be numbers."""
    return df[columns].apply(pd.to_numeric, errors="coerce").describe().T


def values_not_in_reference(df: pd.DataFrame, reference: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """For each reference column, the values that never appear in the reference month."""
    return pd.DataFrame([{"column": c, "values_not_in_jan_2012": sorted(set(df[c]) - set(reference[c]))}
                         for c in columns])

### 2.3 Validation
A row is quarantined if any check fails; the reasons are recorded in `quarantine_reason`.

- `town`, `flat_type` and `flat_model` must exist in the Jan 2012 data (exact match).
- `storey_range` must follow the format used in Jan 2012, `NN TO NN` (e.g. `04 TO 06`), with the first floor not
  above the last. The format is checked rather than the exact value, because valid bands change over time
  (Mar-May 2012 used 5-floor bands such as `01 TO 05`, and taller blocks add bands such as `40 TO 42`).
- No empty values; numeric columns must be valid positive numbers.
- Exact duplicates: the first copy is kept, the others are quarantined.

In [6]:
def invalid_storey_range(storey: pd.Series) -> pd.Series:
    """True where the value is not 'NN TO NN' or the first floor is above the last."""
    floors = storey.str.extract(STOREY_PATTERN).astype(float)
    return floors.isna().any(axis=1) | (floors[0] > floors[1])


def validation_checks(df: pd.DataFrame, reference: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """One True/False column per check (True = failed)."""
    checks = pd.DataFrame(index=df.index)
    for column in REFERENCE_COLUMNS:
        checks[f"{column}_not_in_jan_2012"] = ~df[column].isin(set(reference[column]))
    checks["invalid_storey_range"] = invalid_storey_range(df["storey_range"])
    checks["empty_value"] = (df[columns].apply(lambda s: s.str.strip()) == "").any(axis=1)
    numbers = df[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
    checks["invalid_number"] = (numbers.isna() | (numbers <= 0)).any(axis=1)
    checks["duplicate"] = df.duplicated(subset=columns, keep="first")
    for check, failed in checks.sum().items():
        log_raw_to_silver.info("Check %-28s %6d rows failed", check, failed)
    return checks


def split_valid_invalid(df: pd.DataFrame, checks: pd.DataFrame):
    """Return (passed, quarantined); quarantined rows list every failed check."""
    failed = checks.any(axis=1)
    reasons = checks[failed].apply(lambda row: ";".join(row.index[row]), axis=1)
    return df[~failed].copy(), df[failed].assign(quarantine_reason=reasons)

### 2.4 Cleaning, remaining lease and price anomalies
1. Convert numeric columns to numbers (validation has already confirmed they are valid).
2. **Remaining lease** at 1 Jan 2027 for a 99-year lease, rounded down to whole years and months. The data only
   gives the year the lease started, so the lease is assumed to start on 1 January of that year.
3. **Price anomalies:** price per sqm is compared with similar sales (same town, flat type and year). A sale
   outside Q1 − 3×IQR / Q3 + 3×IQR is flagged `high` or `low`; groups with fewer than 30 sales are `not_assessed`.
   Anomalies are flagged, not removed.

In [7]:
def cast_types(df: pd.DataFrame) -> pd.DataFrame:
    return df.astype({"floor_area_sqm": float, "resale_price": float, "lease_commence_date": int})


def add_remaining_lease(df: pd.DataFrame, as_of: pd.Timestamp, lease_years: int) -> pd.DataFrame:
    """Whole months from as_of to the lease end (1 Jan of start year + lease_years), floored, never negative."""
    lease_end_month = (df["lease_commence_date"] + lease_years) * 12           # 1 January of the end year
    as_of_month = as_of.year * 12 + (as_of.month - 1) + (1 if as_of.day > 1 else 0)
    months = (lease_end_month - as_of_month).clip(lower=0)
    return df.assign(remaining_lease_years=months // 12, remaining_lease_months=months % 12)


def flag_price_anomalies(df: pd.DataFrame) -> pd.DataFrame:
    """Flag price per sqm outside the Tukey fences of its peer group."""
    df = df.assign(sale_year=df["month"].str[:4],
                   price_per_sqm=(df["resale_price"] / df["floor_area_sqm"]).round(2))
    group = df.groupby(ANOMALY_GROUP)["price_per_sqm"]
    q1, q3 = group.transform("quantile", 0.25), group.transform("quantile", 0.75)
    iqr = q3 - q1
    df["price_anomaly"] = np.select(
        [group.transform("size") < MIN_GROUP_SIZE,
         df["price_per_sqm"] > q3 + IQR_MULTIPLIER * iqr,
         df["price_per_sqm"] < q1 - IQR_MULTIPLIER * iqr],
        ["not_assessed", "high", "low"], default="normal")
    log_raw_to_silver.info("Price anomalies: %s", df["price_anomaly"].value_counts().to_dict())
    return df.drop(columns="sale_year")

### 2.5 Run the stage
Load → combine → profile → validate → clean → write the silver layer. The stage checks that every in-scope row
ends up in exactly one of Cleaned or Quarantined.

In [8]:
def raw_to_silver() -> dict:
    """Stage 2: raw files -> silver/cleaned.csv and silver/quarantined.csv."""
    raw = load_raw_files(RAW_DIR)
    master, columns = combine(raw, START_MONTH, END_MONTH, REFERENCE_MONTH)
    reference = master[master["month"] == REFERENCE_MONTH]

    profile = {
        "columns": column_profile(master[columns]),
        "numbers": numeric_profile(master, NUMERIC_COLUMNS),
        "sales_per_year": master["month"].str[:4].value_counts().sort_index().rename("rows").to_frame(),
        "not_in_jan_2012": values_not_in_reference(master, reference, REFERENCE_COLUMNS),
        "storey_ranges": master["storey_range"].value_counts().sort_index().rename("rows").to_frame(),
    }

    if invalid_storey_range(reference["storey_range"]).any():
        raise ValueError("Jan 2012 storey ranges do not fit the configured storey pattern")
    checks = validation_checks(master, reference, columns)
    passed, quarantined = split_valid_invalid(master, checks)
    if len(passed) + len(quarantined) != len(master):
        raise ValueError("Rows were lost during validation")

    cleaned = flag_price_anomalies(add_remaining_lease(cast_types(passed), LEASE_AS_OF, LEASE_YEARS))
    cleaned.to_csv(SILVER_DIR / "cleaned.csv", index=False)
    quarantined.to_csv(SILVER_DIR / "quarantined.csv", index=False)
    log_raw_to_silver.info("Wrote %s (%d rows) and %s (%d rows)", SILVER_DIR / "cleaned.csv", len(cleaned),
                           SILVER_DIR / "quarantined.csv", len(quarantined))
    return {"raw": raw, "master": master, "profile": profile, "checks": checks,
            "cleaned": cleaned, "quarantined": quarantined}


silver = run_stage("raw_to_silver", raw_to_silver)

2026-10-14 20:53:44 | raw_to_silver  | INFO    | Stage started


2026-10-14 20:53:44 | raw_to_silver  | INFO    | Read d_2d5ff9ea31397b66239f245f57751537.csv: 52203 rows, 10 columns


2026-10-14 20:53:44 | raw_to_silver  | INFO    | Read d_43f493c6c50d54243cc1eab0df142d6a.csv: 369651 rows, 10 columns


2026-10-14 20:53:45 | raw_to_silver  | INFO    | Read d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv: 242454 rows, 11 columns


2026-10-14 20:53:45 | raw_to_silver  | INFO    | Read d_ea9ed51da2787afaf8e51f827c304208.csv: 37153 rows, 11 columns


...

2026-10-14 20:53:45 | raw_to_silver  | INFO    | Check invalid_number                    0 rows failed


2026-10-14 20:53:45 | raw_to_silver  | INFO    | Check duplicate                       451 rows failed


2026-10-14 20:53:45 | raw_to_silver  | INFO    | Price anomalies: {'normal': 109204, 'not_assessed': 2146, 'high': 378, 'low': 7}


2026-10-14 20:53:46 | raw_to_silver  | INFO    | Wrote data/silver/cleaned.csv (111735 rows) and data/silver/quarantined.csv (1318 rows)


2026-10-14 20:53:46 | raw_to_silver  | INFO    | Stage finished in 1.3 s


### 2.6 Results
**Raw files and profiling**

In [9]:
pd.DataFrame([{"file": name, "rows": len(df), "columns": ", ".join(df.columns),
               "first_month": df["month"].min(), "last_month": df["month"].max()}
              for name, df in silver["raw"].items()])

,file,rows,columns,first_month,last_month
0,d_2d5ff9ea31397b66239f245f57751537.csv,52203,"month, town, flat_type, block, street_name, st...",2012-03,2014-12
1,d_43f493c6c50d54243cc1eab0df142d6a.csv,369651,"month, town, flat_type, block, street_name, st...",2000-01,2012-02
2,d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv,242454,"month, town, flat_type, block, street_name, st...",2017-01,2026-10
3,d_ea9ed51da2787afaf8e51f827c304208.csv,37153,"month, town, flat_type, block, street_name, st...",2015-01,2016-12


In [10]:
silver["profile"]["columns"]

,column,empty,distinct,most_common
0,month,0,72,2012-03
1,town,0,26,JURONG WEST
2,flat_type,0,7,4 ROOM
3,block,0,2193,2
4,street_name,0,526,YISHUN RING RD
5,storey_range,0,25,04 TO 06
6,floor_area_sqm,0,169,67
7,flat_model,0,20,Model A
8,lease_commence_date,0,50,1985
9,resale_price,0,2865,400000


In [11]:
silver["profile"]["numbers"]

,count,mean,std,min,25%,50%,75%,max
floor_area_sqm,113053.0,96.783885,24.559078,31.0,74.0,96.0,111.0,280.0
resale_price,113053.0,449659.945939,132259.221422,175000.0,354000.0,425000.0,515000.0,1180000.0
lease_commence_date,113053.0,1990.553873,10.757369,1966.0,1983.0,1988.0,2000.0,2015.0


In [12]:
master = silver["master"]
print(silver["profile"]["sales_per_year"].T.to_string())
print(f"\nExact duplicate rows (beyond the first copy): {silver['checks']['duplicate'].sum():,}")
silver["profile"]["not_in_jan_2012"]

month   2012   2013   2014   2015   2016   2017
rows   23198  16097  16096  17780  19373  20509

Exact duplicate rows (beyond the first copy): 451


,column,values_not_in_jan_2012
0,town,[]
1,flat_type,[]
2,flat_model,"[2-room, DBSS, Improved-Maisonette, Premium Ap..."


In [13]:
# storey_range is validated by format, not by value: every band in the data, and how many sales use it
silver["profile"]["storey_ranges"].T

storey_range,01 TO 03,01 TO 05,04 TO 06,06 TO 10,07 TO 09,10 TO 12,11 TO 15,13 TO 15,16 TO 18,16 TO 20,19 TO 21,21 TO 25,22 TO 24,25 TO 27,26 TO 30,28 TO 30,31 TO 33,31 TO 35,34 TO 36,36 TO 40,37 TO 39,40 TO 42,43 TO 45,46 TO 48,49 TO 51
rows,21036,2700,25934,2474,23193,19986,1259,8657,3518,265,1533,92,1010,536,39,347,121,2,132,7,125,60,11,11,5


**Validation**

In [14]:
silver["checks"].sum().rename("rows_failed").to_frame()

,rows_failed
town_not_in_jan_2012,0
flat_type_not_in_jan_2012,0
flat_model_not_in_jan_2012,870
invalid_storey_range,0
empty_value,0
invalid_number,0
duplicate,451


In [15]:
# Quarantined rows by year
pd.DataFrame({"rows": master["month"].str[:4].value_counts(),
              "quarantined": silver["quarantined"]["month"].str[:4].value_counts()}).fillna(0).astype(int).sort_index()

,rows,quarantined
month,,
2012,23198,17
2013,16097,14
2014,16096,301
2015,17780,175
2016,19373,257
2017,20509,554


**Remaining lease and price anomalies**

In [16]:
cleaned = silver["cleaned"]
cleaned[["month", "town", "flat_type", "lease_commence_date", "remaining_lease_years",
         "remaining_lease_months", "price_per_sqm", "price_anomaly"]].head()

,month,town,flat_type,lease_commence_date,remaining_lease_years,remaining_lease_months,price_per_sqm,price_anomaly
0,2012-03,ANG MO KIO,2 ROOM,1986,58,0,5555.56,not_assessed
1,2012-03,ANG MO KIO,2 ROOM,1980,52,0,6022.73,not_assessed
2,2012-03,ANG MO KIO,3 ROOM,1980,52,0,4632.35,normal
3,2012-03,ANG MO KIO,3 ROOM,1984,56,0,4776.12,normal
4,2012-03,ANG MO KIO,3 ROOM,1980,52,0,4791.04,normal


In [17]:
# How many sales were flagged, and the most extreme ones
print(cleaned["price_anomaly"].value_counts().to_string())
cleaned[cleaned["price_anomaly"].isin(["high", "low"])].sort_values("price_per_sqm")[
    ["month", "town", "flat_type", "block", "flat_model", "floor_area_sqm", "resale_price",
     "price_per_sqm", "price_anomaly"]].iloc[[0, 1, 2, -3, -2, -1]]

price_anomaly
normal          109204
not_assessed      2146
high               378
low                  7


,month,town,flat_type,block,flat_model,floor_area_sqm,resale_price,price_per_sqm,price_anomaly
16660,2012-10,YISHUN,4 ROOM,279,Model A,104.0,247000.00,2375.00,low
60551,2017-04,PASIR RIS,4 ROOM,560,Model A,110.0,311000.00,2827.27,low
71665,2017-10,PUNGGOL,EXECUTIVE,112,Premium Apartment,130.0,480000.00,3692.31,low
81566,2015-05,BUKIT MERAH,3 ROOM,48,Standard,60.0,613000.00,10216.67,high
17720,2012-11,KALLANG/WHAMPOA,3 ROOM,56,Terrace,78.0,812000.00,10410.26,high
58859,2017-03,QUEENSTOWN,3 ROOM,55,Terrace,84.0,888888.88,10582.01,high


## 3. Stage `silver_to_gold` — Cleaned → Transformed + Hashed
This stage reads `data/silver/cleaned.csv` (as text, so values are written to gold exactly as they are in silver)
and writes the gold layer.

### 3.1 Resale identifier
Format: `S` + first 3 digits of the block (zero-padded, letters removed) + first 3 digits of the average resale
price for that month, town and flat type + YYMM + first letter of the town. Example: block 19, average $230,000,
Jan 2012, Ang Mo Kio → `S0192301201A`.

Several sales can share this identifier (same block, month and flat type), so a 2-digit sequence is added to every
identifier (`…01`, `…02`). Rows are numbered after sorting by their own values, so the result does not depend on the
order of the source files.

In [18]:
def base_identifier(df: pd.DataFrame) -> pd.Series:
    block = df["block"].str.replace(r"\D", "", regex=True).str[:3].str.zfill(3)
    average_price = pd.to_numeric(df["resale_price"]).groupby([df["month"], df["town"], df["flat_type"]]).transform("mean")
    price = average_price.round(2).astype(int).astype(str).str[:3]
    year_month = df["month"].str[2:4] + df["month"].str[5:7]
    return "S" + block + price + year_month + df["town"].str[0]


def add_resale_identifier(df: pd.DataFrame, sort_columns: list[str]) -> pd.DataFrame:
    """Base identifier plus a 2-digit sequence that makes every identifier unique."""
    base = base_identifier(df)
    log_silver_to_gold.info("%d rows, %d distinct base identifiers (%d rows share one)",
                            len(df), base.nunique(), base.duplicated(keep=False).sum())
    df = df.assign(base_id=base).sort_values(["base_id", *sort_columns], kind="stable")
    sequence = df.groupby("base_id").cumcount() + 1
    if sequence.max() > 99:
        raise ValueError("More than 99 sales share one identifier: the 2-digit sequence is too short")
    df.insert(0, "resale_identifier", df["base_id"] + sequence.astype(str).str.zfill(2))
    return df.drop(columns="base_id").sort_index()

### 3.2 Hash the identifier
The identifier can be rebuilt from public data, so a plain hash (e.g. SHA-256) could be reversed by hashing every
possible identifier and comparing — a dictionary attack. **HMAC-SHA256 with a secret key** prevents this: without
the key nobody can produce a matching hash. It is one-way, and different identifiers give different hashes.

The key is read from the environment variable `RESALE_ID_HMAC_KEY`. If it is not set, a random key is generated for
this run (the hashes will then differ between runs) and a warning is logged. Set a key with:
`export RESALE_ID_HMAC_KEY=$(python -c "import secrets; print(secrets.token_hex(32))")`

In [19]:
def get_hmac_key() -> bytes:
    key = os.getenv(HMAC_KEY_ENV)
    if not key:
        log_silver_to_gold.warning("%s not set: using a random key, hashes will differ between runs", HMAC_KEY_ENV)
        key = secrets.token_hex(32)
    return key.encode()


def hash_identifier(identifiers: pd.Series, key: bytes) -> pd.Series:
    return identifiers.map(lambda value: hmac.new(key, value.encode(), hashlib.sha256).hexdigest())

### 3.3 Run the stage

In [20]:
def silver_to_gold() -> dict:
    """Stage 3: silver/cleaned.csv -> gold/transformed.csv and gold/hashed.csv."""
    cleaned_path = SILVER_DIR / "cleaned.csv"
    if not cleaned_path.exists():
        raise FileNotFoundError(f"{cleaned_path} not found: run stage raw_to_silver first")
    cleaned = pd.read_csv(cleaned_path, dtype=str, keep_default_na=False)
    log_silver_to_gold.info("Read %s: %d rows", cleaned_path, len(cleaned))

    source_columns = list(cleaned.columns[:cleaned.columns.get_loc("source_file")])  # the Jan 2012 schema
    transformed = add_resale_identifier(cleaned, source_columns)
    if not transformed["resale_identifier"].is_unique:
        raise ValueError("Resale identifiers are not unique")

    hashed = transformed.copy()
    hashed["resale_identifier"] = hash_identifier(transformed["resale_identifier"], get_hmac_key())
    if not hashed["resale_identifier"].is_unique:
        raise ValueError("Hashed identifiers are not unique")

    transformed.to_csv(GOLD_DIR / "transformed.csv", index=False)
    hashed.to_csv(GOLD_DIR / "hashed.csv", index=False)
    log_silver_to_gold.info("Wrote %s and %s (%d rows each)", GOLD_DIR / "transformed.csv",
                            GOLD_DIR / "hashed.csv", len(hashed))
    return {"transformed": transformed, "hashed": hashed}


gold = run_stage("silver_to_gold", silver_to_gold)

2026-10-14 20:53:46 | silver_to_gold | INFO    | Stage started


2026-10-14 20:53:46 | silver_to_gold | INFO    | Read data/silver/cleaned.csv: 111735 rows


2026-10-14 20:53:46 | silver_to_gold | INFO    | 111735 rows, 97657 distinct base identifiers (25661 rows share one)


2026-10-14 20:53:46 | silver_to_gold | WARNING | RESALE_ID_HMAC_KEY not set: using a random key, hashes will differ between runs


2026-10-14 20:53:47 | silver_to_gold | INFO    | Wrote data/gold/transformed.csv and data/gold/hashed.csv (111735 rows each)


2026-10-14 20:53:47 | silver_to_gold | INFO    | Stage finished in 1.0 s


### 3.4 Results

In [21]:
gold["transformed"][["resale_identifier", "month", "town", "flat_type", "block", "resale_price"]].head()

,resale_identifier,month,town,flat_type,block,resale_price
0,S1722571203A01,2012-03,ANG MO KIO,2 ROOM,172,250000.0
1,S5102571203A01,2012-03,ANG MO KIO,2 ROOM,510,265000.0
2,S6103651203A01,2012-03,ANG MO KIO,3 ROOM,610,315000.0
3,S4743651203A01,2012-03,ANG MO KIO,3 ROOM,474,320000.0
4,S6043651203A02,2012-03,ANG MO KIO,3 ROOM,604,321000.0


In [22]:
gold["hashed"][["resale_identifier", "month", "town", "flat_type", "resale_price"]].head()

,resale_identifier,month,town,flat_type,resale_price
0,65547d6020419d416ec56271c1430cbe8675c45edb1e9f...,2012-03,ANG MO KIO,2 ROOM,250000.0
1,c775b1ae35388157fad7d2e84fae1d0647d4a8a7491714...,2012-03,ANG MO KIO,2 ROOM,265000.0
2,27ff07ac0a83f119f593ebb68bcf2461478cfeb6968266...,2012-03,ANG MO KIO,3 ROOM,315000.0
3,050a81745e4e46c69c22c8f6fc5a416f35c55707ba6228...,2012-03,ANG MO KIO,3 ROOM,320000.0
4,e0951127e439f6e8a933595a18fbc076ddc2ee71da804b...,2012-03,ANG MO KIO,3 ROOM,321000.0


## 4. Summary
Row counts of every output group, read back from the files on disk.

In [23]:
def rows_in(path: Path) -> int:
    return len(pd.read_csv(path, dtype=str, keep_default_na=False))


summary = pd.DataFrame([
    {"stage": "src_to_raw", "output group": "Raw", "location": str(RAW_DIR),
     "rows": sum(rows_in(p) for p in RAW_DIR.glob("*.csv"))},
    {"stage": "raw_to_silver", "output group": "Cleaned", "location": str(SILVER_DIR / "cleaned.csv"),
     "rows": rows_in(SILVER_DIR / "cleaned.csv")},
    {"stage": "raw_to_silver", "output group": "Quarantined", "location": str(SILVER_DIR / "quarantined.csv"),
     "rows": rows_in(SILVER_DIR / "quarantined.csv")},
    {"stage": "silver_to_gold", "output group": "Transformed", "location": str(GOLD_DIR / "transformed.csv"),
     "rows": rows_in(GOLD_DIR / "transformed.csv")},
    {"stage": "silver_to_gold", "output group": "Hashed", "location": str(GOLD_DIR / "hashed.csv"),
     "rows": rows_in(GOLD_DIR / "hashed.csv")},
])
print(f"Full log: {LOG_FILE}")
summary

Full log: logs/pipeline.log


,stage,output group,location,rows
0,src_to_raw,Raw,data/raw,701461
1,raw_to_silver,Cleaned,data/silver/cleaned.csv,111735
2,raw_to_silver,Quarantined,data/silver/quarantined.csv,1318
3,silver_to_gold,Transformed,data/gold/transformed.csv,111735
4,silver_to_gold,Hashed,data/gold/hashed.csv,111735
